In [1]:
!pip install pydantic openai openai-agents -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.0/843.0 kB 494.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 423.7 kB/s eta 0:00:00


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your_openai_api_key"

In [ ]:
from agents import WebSearchTool, Agent, ModelSettings, TResponseInputItem, Runner, RunConfig
from pydantic import BaseModel
from openai.types.shared.reasoning import Reasoning

# Tool definitions
web_search_preview = WebSearchTool(
  user_location={
    "type": "approximate",
    "country": None,
    "region": None,
    "city": None,
    "timezone": None
  },
  search_context_size="medium"
)
class StockResearchAgentSchema(BaseModel):
  research_summary: str


class TradeDecisionAgentSchema(BaseModel):
  stock: str
  current_view: str
  suggested_action: str
  key_reasons: str
  risk_factors: str


stock_research_agent = Agent(
  name="Stock Research Agent",
  instructions="""You are a helpful and analytical stock research assistant.
When a user asks about any company or stock, you must:

Search the internet for the latest and most relevant information such as news, earnings, forecasts, stock price, and analyst sentiment.
Summarize clearly in plain, factual language without financial jargon.
Highlight key insights, including:

Current stock price and recent trend
Company overview (sector, products, market position)
Financial highlights (revenue, profit, P/E ratio, market cap)
Recent news or analyst ratings
Short-term outlook or risks

Always rely on recent and trustworthy sources such as Yahoo Finance, CNBC, Bloomberg, or Reuters.
Maintain an objective, fact-based tone and never provide personal investment advice.
If no reliable data is available, clearly state that information could not be found.""",
  model="gpt-5",
  tools=[
    web_search_preview
  ],
  output_type=StockResearchAgentSchema,
  model_settings=ModelSettings(
    store=True,
    reasoning=Reasoning(
      effort="low"
    )
  )
)


trade_decision_agent = Agent(
  name="Trade Decision Agent",
  instructions="""You are a disciplined and analytical trade decision agent.
You receive summarized research reports about stocks or markets and must:

Analyze the provided research summary to assess bullish, bearish, or neutral sentiment.
Evaluate technical, fundamental, and news-based indicators where available.
Form a clear trade recommendation, such as:

Buy / Accumulate
Hold / Wait
Sell / Exit

Support your recommendation with short, factual reasoning (e.g., valuation trend, earnings growth, risk signals).
Maintain a professional, risk-aware, and objective tone.
Do not give financial advice to individuals — only provide a general market-oriented trade assessment.
If the research summary lacks sufficient data, clearly state that the decision cannot be made confidently.

Decision Output Format:

Stock: [Name / Symbol]
Current View: [Bullish / Bearish / Neutral]
Suggested Action: [Buy / Hold / Sell]
Key Reasons: [Brief explanation]
Risk Factors: [If applicable]""",
  model="gpt-5",
  output_type=TradeDecisionAgentSchema,
  model_settings=ModelSettings(
    store=True,
    reasoning=Reasoning(
      effort="low"
    )
  )
)


class WorkflowInput(BaseModel):
  input_as_text: str


# Main code entrypoint
async def run_workflow(workflow_input: WorkflowInput):
  workflow = workflow_input.model_dump()
  conversation_history: list[TResponseInputItem] = [
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": workflow["input_as_text"]
        }
      ]
    }
  ]
  stock_research_agent_result_temp = await Runner.run(
    stock_research_agent,
    input=[
      *conversation_history
    ],
    run_config=RunConfig(trace_metadata={
      "__trace_source__": "agent-builder",
      "workflow_id": "wf_68e6759466448190bba97f1fcb9234f404aa1f193e660490"
    })
  )

  conversation_history.extend([item.to_input_item() for item in stock_research_agent_result_temp.new_items])

  stock_research_agent_result = {
    "output_text": stock_research_agent_result_temp.final_output.json(),
    "output_parsed": stock_research_agent_result_temp.final_output.model_dump()
  }
  trade_decision_agent_result_temp = await Runner.run(
    trade_decision_agent,
    input=[
      *conversation_history
    ],
    run_config=RunConfig(trace_metadata={
      "__trace_source__": "agent-builder",
      "workflow_id": "wf_68e6759466448190bba97f1fcb9234f404aa1f193e660490"
    })
  )

  conversation_history.extend([item.to_input_item() for item in trade_decision_agent_result_temp.new_items])

  trade_decision_agent_result = {
    "output_text": trade_decision_agent_result_temp.final_output.json(),
    "output_parsed": trade_decision_agent_result_temp.final_output.model_dump()
  }
  return trade_decision_agent_result


In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import asyncio

async def main():
    print("📈 Stock Research & Trade Decision Bot (type 'exit' to quit)\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "bye"]:
            print("Bot: Goodbye!")
            break

        wf_input = WorkflowInput(input_as_text=user_input)
        result = await run_workflow(wf_input)

        output = result["output_parsed"]
        print("\n🔹 Stock:", output["stock"])
        print("🔹 Current View:", output["current_view"])
        print("🔹 Suggested Action:", output["suggested_action"])
        print("🔹 Key Reasons:", output["key_reasons"])
        print("🔹 Risk Factors:", output["risk_factors"])
        print("-" * 50, "\n")

asyncio.run(main())

📈 Stock Research & Trade Decision Bot (type 'exit' to quit)

You: Nvidia

🔹 Stock: NVIDIA (NVDA)
🔹 Current View: Bullish
🔹 Suggested Action: Buy / Accumulate
🔹 Key Reasons: - Strong demand and guidance: Management guided fiscal Q3’26 revenue to ~$54B ±2% with elevated margins, reflecting continued AI infrastructure demand and Blackwell ramp. [turn0search4]
- Solid profitability and scale: TTM revenue ≈ $165B, net margin ~52%, gross margin ~70%, indicating robust operating leverage. [turn1search0]
- Market leadership: Dominant position in AI GPUs, networking, and CUDA software creates ecosystem advantages and pricing power. [turn1search0]
- Momentum and sentiment: YTD +39% with broad Buy/Overweight coverage; stock near 52‑week highs, supported by ongoing datacenter spend. [turn0finance0][turn1search0]
🔹 Risk Factors: - Valuation sensitivity: Rich multiples (TTM P/E ~53; forward ~33) heighten drawdown risk if growth slows. [turn1search0]
- Supply and customer timing: Capacity and data‑ce